In [1]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

conn = sqlite3.connect(":memory:")

customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

print("Data Loaded Successfully!")

Data Loaded Successfully!


In [2]:
query1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""

pd.read_sql_query(query1, conn).head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


In [3]:
queryA = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

pd.read_sql_query(queryA, conn)

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1
...,...,...
69,WANDK,2
70,WARTH,6
71,WELLI,1
72,WHITC,7


In [4]:
queryB = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

pd.read_sql_query(queryB, conn)

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34
5,ERNSH,6205.39
6,FOLIG,637.94
7,FOLKO,1678.08
8,FRANK,1403.44
9,GODOS,568.27


In [ ]:
WHERE filters rows before grouping, so only orders with Freight > 50 are counted.

HAVING filters after grouping, so it considers total freight per customer and then applies the condition.

Hence, results differ because one works on individual rows and the other on grouped data.

In [5]:
query_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

pd.read_sql_query(query_inner, conn)

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52
...,...,...,...,...
84,Wartian Herkku,Finland,15,822.48
85,Wellington Importadora,Brazil,9,194.71
86,White Clover Markets,USA,14,1353.06
87,Wilman Kala,Finland,7,88.41


In [6]:
query_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight), 0) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID
"""

pd.read_sql_query(query_left, conn)

,companyName,country,order_count,total_freight
0,Alfreds Futterkiste,Germany,6,225.58
1,Ana Trujillo Emparedados y helados,Mexico,4,97.42
2,Antonio Moreno Taquería,Mexico,7,268.52
3,Around the Horn,UK,13,471.95
4,Berglunds snabbköp,Sweden,18,1559.52
...,...,...,...,...
86,Wartian Herkku,Finland,15,822.48
87,Wellington Importadora,Brazil,9,194.71
88,White Clover Markets,USA,14,1353.06
89,Wilman Kala,Finland,7,88.41


INNER JOIN includes only customers with orders.

LEFT JOIN includes all customers, even those without orders.

So, LEFT JOIN shows more rows with NULL or 0 values.